<a href="https://colab.research.google.com/github/lxndrbnsv/cir-notebooks/blob/main/CIR_calls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Зависимости

In [ ]:
!pip install faster-whisper pyannote.audio torchaudio
!pip install -q reportlab
!apt-get install ffmpeg
!apt-get install -q fonts-dejavu

In [ ]:
import io
import os
import re
from datetime import datetime
from collections import Counter

from tqdm import tqdm
from google.colab import files
from google.colab import userdata
from IPython.display import display_markdown

import openai
from huggingface_hub import login

import torch
import torchaudio
from pyannote.audio import Pipeline
from faster_whisper import WhisperModel
import soundfile as sf

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

# Константы

In [ ]:
HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
LLM_TOKEN = userdata.get("LLM_TOKEN")

In [ ]:
LLM_MODEL = "route-llm"

In [ ]:
LLM_BASE_URL = "https://routellm.abacus.ai/v1"

In [ ]:
PROMPT = """Ты — эксперт по анализу звонков отдела продаж.
Тебе предоставлен набор транскриптов телефонных разговоров одного менеджера с клиентами.

ВАЖНО — ПРАВИЛА ФОРМАТИРОВАНИЯ ОТВЕТА:
- Заголовки оформляй через ## и ###
- Жирный текст только через **текст**
- Списки только через "- " (дефис и пробел)
- Не используй: ═══, ───, *текст*, _текст_, нумерованные списки с точкой (1. 2. 3.)
- Разделы отбивай пустой строкой

Твоя задача состоит из двух частей: разбор каждого звонка отдельно, затем сводный отчёт.

## ЧАСТЬ 1. АНАЛИЗ КАЖДОГО ЗВОНКА

Обязательно разбери КАЖДЫЙ звонок из транскриптов — не пропускай ни одного.

Сначала определи тип звонка:
- **Продажа** — менеджер предлагает услуги, обсуждает сотрудничество, работает с клиентом
- **Не продажа** — организационный вопрос, перенос встречи, уточнение деталей, прочее

Для каждого звонка используй соответствующую структуру:

### Если звонок — ПРОДАЖА:

## ЗВОНОК [номер] — [имя клиента или краткая тема]

**Кратко:** с кем разговор и о чём.

### Выявление потребностей
- Задавал ли менеджер вопросы для выявления потребностей клиента?
- Какие потребности удалось выявить?
- Что осталось невыясненным?

### Работа с возражениями
- Были ли возражения со стороны клиента?
- Как менеджер их отработал?
- Возражения "своя компания", "не надо", "дорого" — встречались ли, как обработаны?

### Точки роста
- Где была возможность продвинуть сделку, но менеджер её упустил?
- Что можно было сказать или спросить, но не было сказано?

### Следующий шаг
- Был ли зафиксирован конкретный следующий шаг (дата, действие)?
- Если нет — что должно было быть предложено?

### Оценка звонка
- Что сделано хорошо, что плохо.
- **Оценка: X/10**

### Если звонок — НЕ ПРОДАЖА:

## ЗВОНОК [номер] — [краткая тема] *(не продажа)*

**Кратко:** суть разговора в 1-2 предложениях.
**Результат:** чем завершился звонок.

### Сильные стороны
- Что менеджер сделал хорошо: тон, чёткость, вежливость, решение вопроса

### Зоны роста
- Что можно было сделать лучше: неточности, затянутость, неловкие моменты

---

## ЧАСТЬ 2. СВОДНЫЙ ОТЧЁТ ПО МЕНЕДЖЕРУ

В сводном отчёте учитывай только звонки-продажи. Остальные не включай в оценку.

### Сильные стороны
- Что менеджер делает стабильно хорошо по всем звонкам-продажам

### Системные слабости
- Какие ошибки повторяются из звонка в звонок

### Топ-3 точки роста
- Конкретные навыки, над которыми нужно работать в первую очередь

### Итоговая оценка
- Средний балл по звонкам-продажам
- Общее впечатление в 2-3 предложениях

Отвечай на русском языке. Будь конкретным — ссылайся на конкретные фразы из транскриптов.

ТРАНСКРИПТЫ:"""

**Авторизация HF**

In [ ]:
login(HF_TOKEN)

# Модели

**Инициализация модели расшифровки**

In [ ]:
model = WhisperModel("large-v3", device="cuda", compute_type="float16")

**Инициализация модели диаризации**

In [ ]:
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1", token=HF_TOKEN
  )
if torch.cuda.is_available():
    pipeline.to(torch.device("cuda"))

# LLM-клиент

In [ ]:
class LLMAPI:
    def __init__(self):
        self.client = openai.OpenAI(
            base_url=LLM_BASE_URL,
            api_key=LLM_TOKEN,
        )

    def analyze(self, data: str) -> str:
        response = self.client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": PROMPT},
                {"role": "user", "content": str(data)},
            ],
        )

        return response.choices[0].message.content

In [ ]:
llm_api = LLMAPI()

# Обработчик PDF

In [ ]:
class MarkdownPDFWriter:
    """Конвертирует markdown-текст в PDF с поддержкой кириллицы."""

    def __init__(self):
        pdfmetrics.registerFont(TTFont('DejaVu', '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'))
        pdfmetrics.registerFont(TTFont('DejaVu-Bold', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'))

        self.normal   = ParagraphStyle('normal', fontName='DejaVu',      fontSize=10, leading=16, spaceAfter=4)
        self.bullet   = ParagraphStyle('bullet', fontName='DejaVu',      fontSize=10, leading=16, spaceAfter=4, leftIndent=10)
        self.heading1 = ParagraphStyle('h1',     fontName='DejaVu-Bold', fontSize=13, leading=20, spaceAfter=6, spaceBefore=10)
        self.heading2 = ParagraphStyle('h2',     fontName='DejaVu-Bold', fontSize=11, leading=18, spaceAfter=4, spaceBefore=8)
        self.filename = None

    def _parse_line(self, text, style):
        text = text.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
        text = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', text)
        return Paragraph(text, style)

    def _build_story(self, text):
        story = []
        for line in text.split('\n'):
            line = line.strip()
            line = re.sub(r'^[═─━=\-]{3,}$', '', line)
            if not line:
                story.append(Spacer(1, 4))
            elif line.startswith('#### '):
                story.append(Paragraph(line[5:], self.heading2))
            elif line.startswith('### '):
                story.append(Paragraph(line[4:], self.heading2))
            elif line.startswith('## '):
                story.append(Paragraph(line[3:], self.heading1))
            elif line.startswith('# '):
                story.append(Paragraph(line[2:], self.heading1))
            elif line.startswith('- '):
                story.append(self._parse_line(line[2:], self.bullet))
            else:
                story.append(self._parse_line(line, self.normal))
        return story

    def save(self, text, filename=None):
        """Генерирует PDF и сохраняет на диск."""
        self.filename = filename or f'analysis_{datetime.now().strftime("%Y%m%d_%H%M%S")}.pdf'
        doc = SimpleDocTemplate(self.filename, pagesize=A4,
                                leftMargin=20*mm, rightMargin=20*mm,
                                topMargin=20*mm, bottomMargin=20*mm)
        doc.build(self._build_story(text))
        print(f'✅ Сохранено: {self.filename}')
        return self

    def download(self):
        """Скачивает сохранённый PDF в Colab."""
        if self.filename is None:
            raise RuntimeError('Сначала вызовите save()')
        files.download(self.filename)
        print(f'📥 Скачивается: {self.filename}')
        return self

In [ ]:
writer = MarkdownPDFWriter()

# Функции

**Функции фильтрации текста**

In [ ]:
def is_junk(text):
    t = text.strip()

    # Пустой
    if not t:
        return True

    # Весь текст в верхнем регистре (звуковые эффекты)
    if t == t.upper() and len(t) > 3:
        return True

    # Содержит латиницу (субтитры, имена авторов и прочие галлюцинации)
    if re.search(r'[A-Za-z]', t):
        return True

    # Очень короткий бессмысленный текст (один звук, артефакт)
    words = t.split()
    if len(words) == 1 and len(t) <= 4:
        return True

    return False

In [ ]:
def filter_transcriptions(transcriptions, max_repeat_ratio=0.15, max_consecutive=1):
    # Считаем сколько раз встречается каждый текст
    text_counts = Counter(t["text"] for t in transcriptions)
    total = len(transcriptions)

    # Тексты которые занимают больше max_repeat_ratio от всех сегментов — мусор
    blacklist = {text for text, count in text_counts.items()
                 if count / total > max_repeat_ratio}

    result = []
    prev_text = None
    consecutive = 0

    for t in transcriptions:
        text = t["text"]

        if text in blacklist:
            prev_text = None
            consecutive = 0
            continue

        if text in blacklist or is_junk(text):
            prev_text = None
            consecutive = 0
            continue

        if text == prev_text:
            consecutive += 1
        else:
            consecutive = 0

        if consecutive > max_consecutive:
            continue

        prev_text = text
        result.append(t)

    return result

**Диаризация**

In [ ]:
def diarize(audio_file):
  # Загружаем аудиофайл
  waveform, sr = torchaudio.load(audio_file)

  # Если стерео -> моно
  if waveform.shape[0] > 1:
      waveform = torch.mean(waveform, dim=0, keepdim=True)

  # Приводим к 16 кГц, если нужно
  if sr != 16000:
      resampler = torchaudio.transforms.Resample(sr, 16000)
      waveform = resampler(waveform)
      sr = 16000

  diarization = pipeline({"waveform": waveform, "sample_rate": sr})

  annotation = diarization.speaker_diarization

  speaker_segments = []
  for turn, _, speaker in annotation.itertracks(yield_label=True):
      speaker_segments.append({
          "start": turn.start,
          "end": turn.end,
          "speaker": speaker
      })

  return speaker_segments

**Извлечение сегмента для транскрибации**

In [ ]:
def extract_segment_buf(audio_path, start, end):
    data, sr = sf.read(audio_path, dtype='float32')
    start_idx = int(start * sr)
    end_idx = int(end * sr)
    seg_data = data[start_idx:end_idx]
    buf = io.BytesIO()
    sf.write(buf, seg_data, sr, format='wav')
    buf.seek(0)
    return buf

**Транскрибация**

In [ ]:
def transcribe(speaker_segments, audio_file):
  transcriptions = []
  for seg in speaker_segments:
      audio_buf = extract_segment_buf(audio_file, seg["start"], seg["end"])
      segments, _ = model.transcribe(
          audio_buf,
          language="ru",
          beam_size=5,
          no_speech_threshold=0.8,
          log_prob_threshold=-1.0,
          compression_ratio_threshold=2.4
      )
      text = " ".join([s.text for s in segments]).strip()
      transcriptions.append({
          "speaker": seg["speaker"],
          "start": seg["start"],
          "end": seg["end"],
          "text": text
      })

  return filter_transcriptions(transcriptions)

# Загрузка файлов

In [ ]:
print('Выберите аудиофайлы для загрузки...')
uploaded = files.upload()

AUDIO_FILES = list(uploaded.keys())
print('✅ Файлы загружены')

# Логика обработки

**Обработанные звонки**

In [ ]:
transcribed = []

**Основной пайплайн**

In [ ]:
for audio_file in tqdm(AUDIO_FILES, desc="Обработка файлов", unit="Файл"):
  tqdm.write(f"Обрабатывается: {audio_file}")
  transcribed.append(transcribe(diarize(audio_file), audio_file))

# ИИ анализ

In [ ]:
result = llm_api.analyze(transcribed)

**Вывод отчета**

In [ ]:
display_markdown(result, raw=True)

**Создание PDF**

In [ ]:
writer.save(result).download()